[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Computer Vision - Diffusion Models

This notebooks trains an image generation model using a Diffusion generative model.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 06/08/2025 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/0130DeepLearningDiffusion.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import pandas as pd

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F
from torchvision.transforms import v2 as TorchVisionTrns
import torchinfo

# Miscellaneous
import os
import random
from time import perf_counter

# Typing
from typing import Callable, Literal, Optional, Tuple, Union
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.manual_seed(seedNum)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

D_CLASSES   = {classIdx: str(classIdx) for classIdx in range(10)}
L_CLASSES   = [str(classIdx) for classIdx in range(10)]
TU_IMG_SIZE = (28, 28, 1)

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DataManipulation import DownloadUrl
from DataVisualization import AnnotateImage, PlotLabelsHistogram, PlotMnistImages

In [ ]:
# General Auxiliary Functions

def TensorImageNumpy( tZ: Tensor ) -> NDArray:
    """Converts an image tensor to a NumPy array."""
    return tZ.squeeze().detach().cpu().numpy()

class MNISTDatasetCSV(torch.utils.data.Dataset):
    """MNIST images and labels from CSV. Diffusion training uses only the images."""

    def __init__( self, csvFilePath: str, subSetType: Literal['All', 'Train', 'Val'], *, tuImgSize: Tuple[int, ...] = (28, 28), hImgTrns: Optional[Callable] = None ) -> None:
        dfData = pd.read_csv(csvFilePath)
        match subSetType:
            case 'All':
                pass
            case 'Train':
                dfData = dfData.iloc[:60000]
            case 'Val':
                dfData = dfData.iloc[60000:]
            case _:
                raise ValueError(f'Unsupported subset type: {subSetType}')
        self._mData = dfData.iloc[:, :-1].to_numpy(np.uint8, copy = True)
        self._vLbl = dfData.iloc[:, -1].to_numpy(np.int64, copy = True)
        self._tuImgSize = tuImgSize
        self._hImgTrns = hImgTrns

    def __len__( self ) -> int:
        return len(self._vLbl)

    def __getitem__( self, idx: int ) -> Tuple[Union[NDArray, Tensor], int]:
        tX = self._mData[idx].reshape(self._tuImgSize).copy()
        if self._hImgTrns is not None:
            tX = self._hImgTrns(tX)
        return tX, int(self._vLbl[idx])

    def GetLabels( self ) -> NDArray:
        return self._vLbl.copy()

    def SetTransform( self, hTrns: Optional[Callable] ) -> None:
        self._hImgTrns = hTrns

* <font color='blue'>(**!**)</font> Go through `MNISTDatasetCSV`. Why does diffusion training not require the digit labels?

## Diffusion Model

A _Diffusion Model_ is a generative model which learns to generate data by reversing a gradual noising process.  
The model is based on two processes:
 - _Forward Process_: Gradually transforms a data sample $\color{cyan}{\boldsymbol{x}}_0$ into noise $\color{green}{\boldsymbol{x}}_T$ by adding a small amount of Gaussian noise at each step.  
   This process is fixed and does not require learning.
 - _Reverse Process_: Gradually transforms noise $\color{green}{\boldsymbol{x}}_T$ into a data sample by removing the noise at each step.  
   The denoising operation is learned from data, usually by predicting the noise added at a given time step.

During training, a clean sample $\color{cyan}{\boldsymbol{x}}_0$ and a random time step $t$ are used to generate a noisy sample $\color{green}{\boldsymbol{x}}_t$.  
The model receives $\color{green}{\boldsymbol{x}}_t$ together with $t$ and is trained to predict the noise used to generate it.  
During inference, the model starts from random noise and repeatedly applies the learned denoising operation until a sample is generated.

Some use cases of _Diffusion Models_:

 - Data Generation  
   Generate new samples which follow the distribution of the training data.
 - Conditional Generation  
   Generate samples according to side information such as a class label, text description, or another image.
 - Image Translation  
   Transform an input image into a corresponding output image while preserving its relevant content.
 - Image Restoration  
   Solve inverse problems such as denoising, inpainting, deblurring, and super resolution.


</br>

* <font color='brown'>(**#**)</font> Unlike an _Auto Encoder_, a Diffusion Model does not generate a sample in a single forward pass.  
  Sampling is an iterative process composed of multiple denoising steps.
* <font color='brown'>(**#**)</font> The number of diffusion steps controls a tradeoff between generation quality and inference time.

### Sampling Diffusion Model

This notebook demonstrates the simplest form of a _Diffusion Model_ for image generation.  
The model learns the distribution of the _MNIST_ data set and generates new digit images by sampling from noise.

The model is composed of:
 - _Forward Process_: Gradually adds Gaussian noise to an MNIST image until its structure is lost.  
   A noise schedule controls the amount of information preserved at each time step.
 - _Denoising Model_: Predicts the noise added to an image at a given time step.  
   In the context of image input, the denoising model is commonly based on a _U-Net_.
 - _Reverse Process_: Starts from Gaussian noise and repeatedly applies the denoising model.  
   Each step removes part of the noise until a digit image is generated.

During training, a clean MNIST image $\color{cyan}{\boldsymbol{x}}_0$ is sampled together with a random time step $t$.  
Noise is added to generate $\color{green}{\boldsymbol{x}}_t$, and the model is trained to predict the sampled noise.  
During inference, the process starts from $\color{green}{\boldsymbol{x}}_T \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I})$ and follows the learned reverse process to generate a new sample.

This notebook demonstrates:
 - Defining a noise schedule for the forward diffusion process.  
   The schedule controls the noise level at every time step.
 - Generating a noisy image at an arbitrary time step.  
   This allows training without iterating through the full forward process.
 - Building a time conditioned denoising model.  
   The model receives both the noisy MNIST image and its diffusion time step.
 - Training the model to predict the added noise.  
   The known noise forms the target of the learning problem.
 - Sampling new MNIST images.  
   The learned denoising model is applied iteratively to transform noise into digit images.

</br>

* <font color='brown'>(**#**)</font> This is an _Unconditional Diffusion Model_: the generated digit is not controlled by a class label or another input.
* <font color='brown'>(**#**)</font> The diffusion process in this notebook operates directly in the image space. _Latent Diffusion Models_ first encode images into a lower dimensional latent space.

In [ ]:
# Parameters

# Data
csvFileName = 'MNIST.csv'
csvFileUrl  = r'https://huggingface.co/datasets/Royi/MNIST/resolve/main/MNIST.csv'

# Model
baseCh       = 32
numDiffSteps = 200
modelName    = 'ModelDiffusionMNIST.pt'

# Training
batchSize    = 128
numWorkers   = 0
numEpochs    = 5
learnRate    = 2e-4
numValBatches = 10

# Visualization
numImg = 16
numPlotSteps = 6

## Intuition on Diffusion Process

An _Auto Encoder_ learns to reconstruct, but does not define where to sample in its latent space. A _VAE_ regularizes that space toward a convenient prior. Diffusion takes another route: fix a gradual transformation from data to Gaussian noise, then learn the reverse transitions.

### From Data to Noise

At each step we shrink the signal and add independent Gaussian noise:

$$ \boldsymbol{x}_t = \sqrt{\alpha_t}\boldsymbol{x}_{t-1} + \sqrt{1-\alpha_t}\boldsymbol{\epsilon}_t, \qquad \alpha_t = 1-\beta_t, \qquad \boldsymbol{\epsilon}_t \sim \mathcal{N}(\boldsymbol{0},\boldsymbol{I}). $$

The accumulated signal retention is $\bar\alpha_t = \prod_{s=1}^{t}\alpha_s$. We can sample any noise level directly:

$$ \boldsymbol{x}_t = \sqrt{\bar\alpha_t}\boldsymbol{x}_0 + \sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I}). $$

As $\bar\alpha_t$ approaches zero, different data distributions approach the same standard Gaussian. The bar denotes a product, not an average.

### From Noise to Data

Learning one small reverse transition is easier than guessing a complete image from pure noise. The same network handles all noise levels, with the time step as an additional input.

We know the noise we added, so it supplies the training target. Under MSE, the network learns the conditional mean of that noise. Its prediction determines the mean of a reverse transition; sampling also accounts for uncertainty.

* <font color='brown'>(**#**)</font> Generation samples a plausible image, not the unique original image hidden in noise.
* <font color='red'>(**?**)</font> Why would predicting a clean image in one step from almost pure noise tend to give an average image?

```mermaid
flowchart TB
    subgraph Forward["Forward process: fixed"]
        direction LR
        Clean["Data sample"] -->|"Shrink + add noise"| Noisy["Noisy sample"]
        Noisy -->|"Repeat"| Gaussian["Approximately standard Gaussian"]
    end
    subgraph Reverse["Reverse process: learned"]
        direction LR
        Fresh["Fresh Gaussian sample"] -->|"Learned reverse transition"| LessNoisy["Less noisy sample"]
        LessNoisy -->|"Repeat"| Generated["New data sample"]
    end
    Forward ~~~ Reverse
```

### Variants of Diffusion Models

Diffusion and flow matching are related approaches to transforming a simple noise distribution into data. 

 * **Score Based Diffusion** - Learns the gradient of the log density at each noise level: a direction toward higher density. Noise-conditioned score models ([2019](https://arxiv.org/abs/1907.05600)) used annealed Langevin sampling; the continuous-time framework supports reverse SDE and probability-flow ODE sampling.
 * **Denoising Diffusion Probabilistic Models (DDPM)** - The [2020 formulation](https://arxiv.org/abs/2006.11239) learns a discrete time reverse noising chain, commonly by predicting added noise.
 * **Flow Matching** - Introduced in [2022](https://arxiv.org/abs/2210.02747), it learns a velocity field along chosen probability paths and generates samples by integrating an ODE. 

* <font color='brown'>(**#**)</font> DDPM uses discrete time steps, but its noisy image values are continuous. This differs from diffusion over discrete states such as tokens.

| Aspect | DDPM (This Notebook) | Score-Based Diffusion | Flow Matching (Straight Path) |
|--------|----------------------|-----------------------|-------------------------------|
| Network Predicts | Added noise $\boldsymbol{\epsilon}$ | Score $\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})$ | Velocity $\boldsymbol{v}(\boldsymbol{x},t)$ |
| Training Target | Sampled noise $\boldsymbol{\epsilon}$ | Conditional score $-\boldsymbol{\epsilon}/\sigma_t$ | Path velocity $\boldsymbol{x}-\boldsymbol{z}$ |
| Training Path | $\boldsymbol{x}_t=a_t\boldsymbol{x}_0+\sigma_t\boldsymbol{\epsilon}$ | Same Gaussian noising path | $\boldsymbol{x}_t=(1-t)\boldsymbol{z}+t\boldsymbol{x}$ |
| Generation | Discrete stochastic reverse steps | Reverse SDE or probability-flow ODE | Integrate the velocity ODE |
| Fresh noise during sampling | Yes, except at the final step | Yes for the SDE; no for the ODE | No, after initialization |
| Time direction used here | Data to noise: $0\rightarrow T$; generate in reverse | Data to noise: increasing time; generate in reverse | Noise to data: $0\rightarrow1$ |

Here $a_t=\sqrt{\bar\alpha_t}$, $\sigma_t=\sqrt{1-\bar\alpha_t}$, $\boldsymbol{z}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I})$, and $\boldsymbol{x}$ is a data sample independent of $\boldsymbol{z}$. SDE and ODE denote stochastic and ordinary differential equations.

* <font color='brown'>(**#**)</font> DDPM and score-based diffusion are closely related: $\boldsymbol{s}_{\theta}(\boldsymbol{x}_t,t)=-\boldsymbol{\epsilon}_{\theta}(\boldsymbol{x}_t,t)/\sigma_t$. Weighting score MSE by $\sigma_t^2$ recovers noise MSE.
* <font color='brown'>(**#**)</font> A time interval does not define the method: diffusion can also use $t\in[0,1]$. The path, prediction target and sampler are the important differences.

In [ ]:
# Forward Diffusion Schedule

class DiffusionSchedule(nn.Module):
    def __init__( self, numSteps: int ) -> None:
        super().__init__()
        self.numSteps = numSteps
        vGrid = torch.linspace(0, 1, numSteps + 1, dtype = torch.float64)
        vCurve = torch.cos((vGrid + 0.008) / 1.008 * np.pi / 2).square()
        vBeta = (1 - vCurve[1:] / vCurve[:-1]).clamp(1e-5, 0.999)
        vAlpha = 1 - vBeta
        vAlphaBar = torch.cumprod(vAlpha, dim = 0)
        vAlphaPrev = torch.cat((torch.ones(1, dtype = torch.float64), vAlphaBar[:-1]))
        self.register_buffer('vBeta', vBeta.float())
        self.register_buffer('vAlphaBar', vAlphaBar.float())
        self.register_buffer('vVariance', (vBeta * (1 - vAlphaPrev) / (1 - vAlphaBar)).float())
        self.register_buffer('vCoefClean', (vBeta * vAlphaPrev.sqrt() / (1 - vAlphaBar)).float())
        self.register_buffer('vCoefNoisy', (vAlpha.sqrt() * (1 - vAlphaPrev) / (1 - vAlphaBar)).float())

    def AddNoise( self, tX: Tensor, vTime: Tensor, tNoise: Tensor ) -> Tensor:
        tAlphaBar = self.vAlphaBar[vTime].view(-1, *([1] * (tX.ndim - 1)))
        return tAlphaBar.sqrt() * tX + (1 - tAlphaBar).sqrt() * tNoise

    def Step( self, tX: Tensor, tNoiseHat: Tensor, stepIdx: int, tNoise: Tensor ) -> Tensor:
        alphaBar = self.vAlphaBar[stepIdx]
        tClean = ((tX - (1 - alphaBar).sqrt() * tNoiseHat) / alphaBar.sqrt()).clamp(-1, 1)
        tMean = self.vCoefClean[stepIdx] * tClean + self.vCoefNoisy[stepIdx] * tX
        return tMean + self.vVariance[stepIdx].sqrt() * tNoise

oDiff = DiffusionSchedule(numDiffSteps)

* <font color='brown'>(**#**)</font> The cosine schedule gradually reduces signal retention. In code, index `0` is the first noisy state ($t=1$); the clean image is $\boldsymbol{x}_0$.
* <font color='brown'>(**#**)</font> `Step()` estimates a clean image, clips that estimate to the known image range, and uses it in the Gaussian reverse mean. Its variance is $\tilde\beta_t=\beta_t(1-\bar\alpha_{t-1})/(1-\bar\alpha_t)$, which is zero at the final step. Noisy states themselves are not clipped.

In [ ]:
# A Distribution Becomes Gaussian

oRng = np.random.default_rng(seedNum)
vAngles = np.linspace(0, 2 * np.pi, 6, endpoint = False)
mCenters = 4 * np.column_stack((np.cos(vAngles), np.sin(vAngles)))
vLabels = np.arange(1800) % 6
mPoints = mCenters[vLabels] + 0.25 * oRng.standard_normal((1800, 2))
vShowSteps = np.linspace(0, numDiffSteps, numPlotSteps, dtype = int)
vBeta = oDiff.vBeta.cpu().numpy()
hF, vHa = plt.subplots(1, numPlotSteps, figsize = (3 * numPlotSteps, 3))

for stepIdx in range(numDiffSteps + 1):
    if stepIdx > 0:
        beta = vBeta[stepIdx - 1]
        mPoints = np.sqrt(1 - beta) * mPoints + np.sqrt(beta) * oRng.standard_normal(mPoints.shape)
    if stepIdx in vShowSteps:
        plotIdx = np.flatnonzero(vShowSteps == stepIdx)[0]
        hA = np.atleast_1d(vHa)[plotIdx]
        hA.scatter(mPoints[:, 0], mPoints[:, 1], c = vLabels, cmap = 'tab10', s = 3, alpha = 0.5)
        hA.set(xlim = (-6, 6), ylim = (-6, 6), title = f'Step {stepIdx}', aspect = 'equal')
hF.tight_layout();

## Generate / Load Data

The data is the MNIST Dataset. This section:

 - Loads the training and validation images.
 - Plots samples and the label distribution.
 - Scales images to $[-1,1]$.
 - Builds the data loaders.

The dataset returns an image and its label. Labels are used only for inspection; the diffusion loop generates its own noise targets.

In [ ]:
# Download Data (CSV)

csvFilePath = os.path.join(DATA_FOLDER_PATH, csvFileName)
csvFilePath = DownloadUrl(csvFileUrl, csvFilePath)

In [ ]:
# Data Set

dsTrain = MNISTDatasetCSV(csvFilePath, 'Train')
dsVal   = MNISTDatasetCSV(csvFilePath, 'Val')

print(f'The number of samples in training data set  : {len(dsTrain)}')
print(f'The number of samples in validation data set: {len(dsVal)}')

In [ ]:
# Element of the Data Set / Data Sample

tX, valY = dsTrain[0]
print(f'The image shape: {tX.shape}')
print(f'The label      : {valY}')

### Plot the Data

In [ ]:
# Plot the Data

mX = np.zeros((9, 28 * 28), dtype = np.uint8)
vY = np.zeros(9, dtype = np.int64)
for sampleIdx in range(9):
    randIdx = random.randint(0, len(dsTrain) - 1)
    tX, valY = dsTrain[randIdx]
    mX[sampleIdx] = tX.flatten()
    vY[sampleIdx] = valY
hF = PlotMnistImages(mX, vY, 3, 3)

In [ ]:
# Plot Single Sample

randIdx = random.randint(0, len(dsTrain) - 1)
tX, valY = dsTrain[randIdx]
hF, hA = plt.subplots(figsize = (7, 7))
hA.imshow(tX, cmap = 'gray', vmin = 0, vmax = 255)
hA.set_title(f'Sample Index: {randIdx}, Label: {valY}')
AnnotateImage(tX, hA, fontSize = 6);

In [ ]:
# Histogram of Labels

hF, vHa = plt.subplots(nrows = 1, ncols = 2, figsize = (8, 4))
vHa = vHa.flat

hA = PlotLabelsHistogram(dsTrain.GetLabels(), hA = vHa[0], lClass = L_CLASSES)
hA.set_title('Histogram of Labels, Training Set');

hA = PlotLabelsHistogram(dsVal.GetLabels(), hA = vHa[1], lClass = L_CLASSES)
hA.set_title('Histogram of Labels, Validation Set');

### Augmentation / Transform

Scale pixels from $[0,255]$ to $[-1,1]$. We omit augmentation to keep the target distribution simple. Diffusion noise is generated in the training loop, not by a dataset transform.

In [ ]:
# Loader Transform

oTrnsTrain = TorchVisionTrns.Compose([
    TorchVisionTrns.ToImage(),
    TorchVisionTrns.ToDtype(torch.float32, scale = True),
    TorchVisionTrns.Normalize(mean = (0.5,), std = (0.5,)),
])
oTrnsVal = oTrnsTrain

In [ ]:
# Apply Transforms

dsTrain.SetTransform(oTrnsTrain)
dsVal.SetTransform(oTrnsVal)

In [ ]:
# Element of the Data Set / Data Sample

tX, valY = dsTrain[0]
print(f'The image shape: {tX.shape}')
print(f'The image range: [{tX.min():.1f}, {tX.max():.1f}]')
print(f'The label      : {valY}')

In [ ]:
# Forward Diffusion of a Single Image

tX, valY = dsVal[23]
tClean = tX.unsqueeze(0).repeat(numPlotSteps, 1, 1, 1)
vTime = torch.linspace(0, numDiffSteps - 1, numPlotSteps).long()
tNoise = torch.randn_like(tClean)
tNoisy = oDiff.AddNoise(tClean, vTime, tNoise)
hF, vHa = plt.subplots(1, numPlotSteps + 1, figsize = (2 * (numPlotSteps + 1), 2))
vHa[0].imshow(TensorImageNumpy(tX), cmap = 'gray', vmin = -1, vmax = 1)
vHa[0].set_title('Clean')
vHa[0].axis('off')
for plotIdx, hA in enumerate(vHa[1:]):
    hA.imshow(TensorImageNumpy(tNoisy[plotIdx]), cmap = 'gray', vmin = -1, vmax = 1)
    hA.set_title(f'Step {vTime[plotIdx].item() + 1}')
    hA.axis('off')
hF.tight_layout();

* <font color='red'>(**?**)</font> What happens if we add noise without attenuating the original signal?
* <font color='brown'>(**#**)</font> The image strip samples each time step directly with independent noise; it is not one sequential trajectory. The display saturates values outside $[-1,1]$, but the noisy tensors remain unclipped.

### Data Loaders

In [ ]:
# Data Loader

dlTrain = torch.utils.data.DataLoader(dsTrain, batch_size = batchSize, shuffle = True, num_workers = numWorkers, pin_memory = torch.cuda.is_available())
dlVal = torch.utils.data.DataLoader(dsVal, batch_size = 2 * batchSize, shuffle = False, num_workers = numWorkers, pin_memory = torch.cuda.is_available())

In [ ]:
# Iterate on the Loader

tX, vY = next(iter(dlTrain))
print(f'The batch image dimensions: {tX.shape}')
print(f'The batch label dimensions: {vY.shape}')

## Build Diffusion Model

The denoiser is a small convolutional _U-Net_: $28\times28 \rightarrow 14\times14 \rightarrow 7\times7$ and back.

 - Low resolution features capture the digit's overall structure.
 - Skip connections preserve spatial detail.
 - A sinusoidal time embedding tells each block the noise level.
 - The output is the predicted noise, with the same shape as the image.

* <font color='brown'>(**#**)</font> There is no classifier or sigmoid: Gaussian noise is not restricted to $[0,1]$.
* <font color='brown'>(**#**)</font> GroupNorm does not depend on batch statistics. Attention and a ViT are unnecessary for this small example.

```mermaid
flowchart TB
    Input["Noisy image: 1 x 28 x 28"] --> Enc1["Encoder 1: 32 x 28 x 28"]
    Enc1 -->|"Average pool"| Enc2["Encoder 2: 64 x 14 x 14"]
    Enc2 -->|"Average pool"| Mid["Middle: 128 x 7 x 7"]
    Mid -->|"Upsample"| Dec2["Decoder 2: 64 x 14 x 14"]
    Enc2 -->|"Skip: concatenate"| Dec2
    Dec2 -->|"Upsample"| Dec1["Decoder 1: 32 x 28 x 28"]
    Enc1 -->|"Skip: concatenate"| Dec1
    Dec1 -->|"1 x 1 convolution"| Output["Predicted noise: 1 x 28 x 28"]
    Time["Time step"] --> Embed["Sinusoidal embedding + MLP"]
    Embed -.-> Enc1
    Embed -.-> Enc2
    Embed -.-> Mid
    Embed -.-> Dec2
    Embed -.-> Dec1
```

Shapes are `channels x height x width` for `baseCh = 32`. Each encoder, middle, and decoder block receives the same time embedding through its own projection.

In [ ]:
# Time Conditioned U-Net

class TimeBlock(nn.Module):
    def __init__( self, inCh: int, outCh: int, timeDim: int ) -> None:
        super().__init__()
        self.oConv = nn.Sequential(nn.Conv2d(inCh, outCh, 3, padding = 1), nn.GroupNorm(8, outCh), nn.SiLU())
        self.oTime = nn.Linear(timeDim, outCh)
        self.oOut = nn.Sequential(nn.GroupNorm(8, outCh), nn.SiLU(), nn.Conv2d(outCh, outCh, 3, padding = 1))
        self.oSkip = nn.Conv2d(inCh, outCh, 1) if inCh != outCh else nn.Identity()

    def forward( self, tX: Tensor, mTime: Tensor ) -> Tensor:
        tZ = self.oConv(tX) + self.oTime(mTime)[:, :, None, None]
        return self.oOut(tZ) + self.oSkip(tX)

class DiffusionUNet(nn.Module):
    def __init__( self, baseCh: int = 32, timeDim: int = 64 ) -> None:
        super().__init__()
        vFreq = torch.exp(-np.log(10000.0) * torch.arange(timeDim // 2) / (timeDim // 2 - 1))
        self.register_buffer('vFreq', vFreq)
        self.oTime = nn.Sequential(nn.Linear(timeDim, timeDim), nn.SiLU(), nn.Linear(timeDim, timeDim))
        self.oEnc1 = TimeBlock(1, baseCh, timeDim)
        self.oEnc2 = TimeBlock(baseCh, 2 * baseCh, timeDim)
        self.oMid = TimeBlock(2 * baseCh, 4 * baseCh, timeDim)
        self.oDec2 = TimeBlock(6 * baseCh, 2 * baseCh, timeDim)
        self.oDec1 = TimeBlock(3 * baseCh, baseCh, timeDim)
        self.oOut = nn.Conv2d(baseCh, 1, 1)

    def forward( self, tX: Tensor, vTime: Tensor ) -> Tensor:
        mAngles = vTime.float()[:, None] * self.vFreq[None, :]
        mTime = self.oTime(torch.cat((mAngles.sin(), mAngles.cos()), dim = 1))
        tEnc1 = self.oEnc1(tX, mTime)
        tEnc2 = self.oEnc2(F.avg_pool2d(tEnc1, 2), mTime)
        tMid = self.oMid(F.avg_pool2d(tEnc2, 2), mTime)
        tDec2 = F.interpolate(tMid, size = tEnc2.shape[-2:], mode = 'nearest')
        tDec2 = self.oDec2(torch.cat((tDec2, tEnc2), dim = 1), mTime)
        tDec1 = F.interpolate(tDec2, size = tEnc1.shape[-2:], mode = 'nearest')
        tDec1 = self.oDec1(torch.cat((tDec1, tEnc1), dim = 1), mTime)
        return self.oOut(tDec1)

In [ ]:
# The Model Object

oModel = DiffusionUNet(baseCh)

In [ ]:
# Model Summary

torchinfo.summary(oModel, input_data = (torch.randn(2, 1, 28, 28), torch.zeros(2, dtype = torch.long)),
                  col_names = ['kernel_size', 'output_size', 'num_params'], device = 'cpu',
                  row_settings = ['depth', 'var_names'])

In [ ]:
# Model Input / Output

with torch.no_grad():
    tTest = torch.randn(2, 1, 28, 28)
    vTestTime = torch.tensor([0, numDiffSteps - 1])
    tNoiseHat = oModel(tTest, vTestTime)
assert tNoiseHat.shape == tTest.shape
print(f'Noisy images: {tTest.shape}, Time steps: {vTestTime.shape}, Predicted noise: {tNoiseHat.shape}')

## Train the Model

Each update samples clean images, a time step per image, and Gaussian noise. The model predicts the sampled noise:

$$ \mathcal{L}(\boldsymbol{w}) = \mathbb{E}_{\boldsymbol{x}_0,t,\boldsymbol{\epsilon}}\left[\left\|\boldsymbol{\epsilon}_{\boldsymbol{w}}(\boldsymbol{x}_t,t)-\boldsymbol{\epsilon}\right\|_2^2\right]. $$

 - Training uses one network evaluation per image, not the entire diffusion chain.
 - Validation uses fixed time steps and noise draws for comparable losses.
 - CUDA uses mixed precision. The best validation checkpoint is saved.

* <font color='brown'>(**#**)</font> Five epochs are a starting budget, not a guaranteed runtime. Target 3-7 minutes on a CUDA GPU; use the measured epoch time to adjust `numEpochs`, then check sample quality. CPU training may take substantially longer.
* <font color='brown'>(**#**)</font> Low noise MSE is useful, but does not by itself guarantee good generated images.

```mermaid
flowchart TB
    Clean["Clean images"] --> AddNoise["Forward formula"]
    Noise["Sample Gaussian noise"] --> AddNoise
    Time["Sample a time per image"] --> AddNoise
    AddNoise -->|"Noisy images"| Model["Time-conditioned U-Net"]
    Time --> Model
    Model -->|"Predicted noise"| Loss["MSE"]
    Noise -->|"Known target"| Loss
    Loss --> Update["Backpropagation + AdamW"]
    Update -.->|"Update weights"| Model
```

One randomly selected noise level per image; no reverse chain is needed during training.

In [ ]:
# Check GPU Availability

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device

In [ ]:
# Noise Prediction Loss

hL = nn.MSELoss()

* <font color='red'>(**?**)</font> Why can the model predict noise better than chance even though the noise was sampled independently of the clean image?
* <font color='brown'>(**#**)</font> Independence from the clean image does not imply independence from the noisy observation. The MSE optimum is $\mathbb{E}[\boldsymbol{\epsilon}\mid\boldsymbol{x}_t,t]$.

In [ ]:
# Training / Validation Epoch

def RunDiffusionEpoch( oModel: nn.Module, oDiff: DiffusionSchedule, dlData, hL: nn.Module, runDevice: torch.device, *, oOpt = None, oScaler = None, maxBatches: Optional[int] = None ) -> float:
    isTrain = oOpt is not None
    oModel.train(isTrain)
    oGen = None if isTrain else torch.Generator(device = runDevice).manual_seed(seedNum + 1)
    totalLoss = torch.zeros((), device = runDevice)
    numSeen = 0

    for batchIdx, (tX, _) in enumerate(dlData):
        if maxBatches is not None and batchIdx >= maxBatches:
            break
        tX = tX.to(runDevice, non_blocking = True)
        vTime = torch.randint(oDiff.numSteps, (len(tX),), device = runDevice, generator = oGen)
        tNoise = torch.randn(tX.shape, device = runDevice, generator = oGen)
        tNoisy = oDiff.AddNoise(tX, vTime, tNoise)
        if isTrain:
            oOpt.zero_grad(set_to_none = True)
        with torch.set_grad_enabled(isTrain):
            with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
                tNoiseHat = oModel(tNoisy, vTime)
                loss = hL(tNoiseHat.float(), tNoise)
            if isTrain:
                if oScaler is not None:
                    oScaler.scale(loss).backward()
                    oScaler.unscale_(oOpt)
                    nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
                    oScaler.step(oOpt)
                    oScaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
                    oOpt.step()
        totalLoss += loss.detach() * len(tX)
        numSeen += len(tX)

    return (totalLoss / numSeen).item()

In [ ]:
# Optimizer and Run Device

oModel = oModel.to(runDevice)
oDiff = oDiff.to(runDevice)
oOpt = torch.optim.AdamW(oModel.parameters(), lr = learnRate, weight_decay = 1e-4)
oScaler = torch.amp.GradScaler('cuda', enabled = runDevice.type == 'cuda')
modelPath = os.path.join(MODELS_FOLDER_PATH, modelName)

In [ ]:
# Training the Model

lTrainLoss, lValLoss, lEpochTime = [], [], []
bestLoss = float('inf')
os.makedirs(MODELS_FOLDER_PATH, exist_ok = True)

for epochIdx in range(numEpochs):
    startTime = perf_counter()
    trainLoss = RunDiffusionEpoch(oModel, oDiff, dlTrain, hL, runDevice, oOpt = oOpt, oScaler = oScaler)
    valLoss = RunDiffusionEpoch(oModel, oDiff, dlVal, hL, runDevice, maxBatches = numValBatches)
    elapsedTime = perf_counter() - startTime
    lTrainLoss.append(trainLoss)
    lValLoss.append(valLoss)
    lEpochTime.append(elapsedTime)
    if valLoss < bestLoss:
        bestLoss = valLoss
        torch.save({'Model': oModel.state_dict(), 'Diffusion': oDiff.state_dict(),
                    'BaseChannels': baseCh, 'NumSteps': numDiffSteps}, modelPath)
    print(f'Epoch {epochIdx + 1:02d}/{numEpochs}: Train MSE = {trainLoss:.4f}, Val MSE = {valLoss:.4f}, Time = {elapsedTime:.1f} s')

print(f'Total training and validation time: {sum(lEpochTime) / 60:.1f} minutes')

In [ ]:
# Plot Training Phase

hF, vHa = plt.subplots(1, 2, figsize = (10, 4))
vEpoch = np.arange(1, len(lTrainLoss) + 1)
vHa[0].plot(vEpoch, lTrainLoss, label = 'Train')
vHa[0].plot(vEpoch, lValLoss, label = 'Validation')
vHa[0].set(title = 'Noise Prediction MSE', xlabel = 'Epoch', ylabel = 'MSE')
vHa[0].legend()
vHa[1].plot(vEpoch, lEpochTime)
vHa[1].set(title = 'Training and Validation Time', xlabel = 'Epoch', ylabel = 'Seconds')
hF.tight_layout();

In [ ]:
# Load the Best Model / Inference Mode

dModel = torch.load(modelPath, map_location = runDevice, weights_only = True)
if dModel['BaseChannels'] != baseCh or dModel['NumSteps'] != numDiffSteps:
    raise ValueError('Checkpoint configuration does not match the notebook parameters.')
oModel.load_state_dict(dModel['Model'])
oDiff.load_state_dict(dModel['Diffusion'])
oModel.eval();

### Generate by Repeated Denoising

```mermaid
flowchart TB
    Start["Sample Gaussian image; set t = T"] --> State["Current image and time"]
    State --> Model["U-Net: predict noise"]
    Model --> Step["Estimate clean image; compute reverse mean"]
    State --> Step
    Step --> Sample["Sample the next image"]
    Noise["Fresh Gaussian noise; zero at final step"] --> Sample
    Sample --> Done{"t = 1?"}
    Done -->|"No"| Next["Use next image; decrease t"]
    Next --> State
    Done -->|"Yes"| Output["Generated image"]
```

Weights stay fixed. The schedule scales the fresh noise by the reverse standard deviation; the final transition adds no noise. Here `t` uses the mathematical `T ... 1` convention; code indices are `T - 1 ... 0`.

In [ ]:
# Reverse Diffusion / DDPM Sampling

@torch.inference_mode()
def SampleDiffusion( oModel: nn.Module, oDiff: DiffusionSchedule, numSamples: int, runDevice: torch.device, sampleSeed: int = 512 ) -> Tuple[Tensor, list, list]:
    oModel.eval()
    oGen = torch.Generator(device = runDevice).manual_seed(sampleSeed)
    tX = torch.randn((numSamples, 1, 28, 28), device = runDevice, generator = oGen)
    lFrames = [tX[:4].cpu().clone()]
    lSteps = [oDiff.numSteps]
    vSaveSteps = np.linspace(oDiff.numSteps, 0, numPlotSteps, dtype = int)

    for stepIdx in reversed(range(oDiff.numSteps)):
        vTime = torch.full((numSamples,), stepIdx, device = runDevice, dtype = torch.long)
        with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
            tNoiseHat = oModel(tX, vTime)
        tNoise = torch.randn(tX.shape, device = runDevice, generator = oGen) if stepIdx > 0 else torch.zeros_like(tX)
        tX = oDiff.Step(tX, tNoiseHat.float(), stepIdx, tNoise)
        if stepIdx in vSaveSteps:
            lFrames.append(tX[:4].cpu().clone())
            lSteps.append(stepIdx)

    return tX.cpu(), lFrames, lSteps

In [ ]:
# Generate New Images from Noise

tSamples, lFrames, lSteps = SampleDiffusion(oModel, oDiff, numImg, runDevice, sampleSeed = seedNum)
numCols = int(np.ceil(np.sqrt(numImg)))
numRows = int(np.ceil(numImg / numCols))
hF, mHa = plt.subplots(numRows, numCols, figsize = (2 * numCols, 2 * numRows), squeeze = False)
for sampleIdx, hA in enumerate(mHa.flat):
    if sampleIdx < numImg:
        hA.imshow(TensorImageNumpy(tSamples[sampleIdx]), cmap = 'gray', vmin = -1, vmax = 1)
    hA.axis('off')
hF.suptitle('Generated MNIST Images')
hF.tight_layout();

In [ ]:
# Follow the Reverse Process

numRows = len(lFrames[0])
hF, mHa = plt.subplots(numRows, len(lFrames), figsize = (2 * len(lFrames), 2 * numRows), squeeze = False)
for frameIdx, (tFrame, stepIdx) in enumerate(zip(lFrames, lSteps)):
    for sampleIdx in range(numRows):
        hA = mHa[sampleIdx, frameIdx]
        hA.imshow(TensorImageNumpy(tFrame[sampleIdx]), cmap = 'gray', vmin = -1, vmax = 1)
        hA.axis('off')
        if sampleIdx == 0:
            hA.set_title(f'Step {stepIdx}')
hF.tight_layout();

* <font color='blue'>(**!**)</font> Change `sampleSeed` and generate new images. No retraining is needed.
* <font color='red'>(**?**)</font> Why do we add noise during reverse sampling, but not at the final step?
* <font color='green'>(**@**)</font> Compare fixed-seed samples after different numbers of training epochs.
* <font color='green'>(**@**)</font> Remove the time embedding, retrain, and compare the results.
* <font color='brown'>(**#**)</font> Do not accelerate this DDPM sampler by simply skipping steps. A sampler such as DDIM uses a different update for larger jumps.